Aim: Clean up geo CSVs (i.e. csvs containing geometry columns) prior to upoad to GEE: removing unwanted tags - artefact of processing led to some tags from other types being present e.g. railway tag in ferry_routes.

In [23]:
import pandas as pd
import os

# Base and feature-specific columns
base_columns = ['id', 'name', 'geometry']

extract_columns = {
    'highways': ['highway', 'surface'],
    'railways': ['railway', 'tunnel', 'bridge', 'electrified', 'usage', 'service', 'layer'],
    'waterways': [
        'waterway', 'boat', 'motorboat', 'canoe', 'kayak', 'sailing', 'navigable', 'navigation',
        'navigation:system', 'usage', 'cemt', 'maxdraft', 'maxlength', 'maxwidth', 'maxheight',
        'width', 'depth', 'mooring', 'lock', 'bridge', 'tunnel', 'layer', 'barrier', 'route',
        'designation', 'class', 'ref', 'operator', 'admin_level', 'intermittent', 'seasonal',
        'ford', 'seamark:type'
    ],
    'boat_access': ['mooring', 'man_made', 'amenity', 'leisure', 'landing', 'natural', 'access', 'operator', 'ref'],
    'ferry_routes': ['route', 'operator', 'duration', 'frequency', 'access', 'layer', 'ref']
}

def detect_type_from_filename(filename):
    filename = filename.lower()
    for key in extract_columns:
        if key in filename:
            return key
    return None

def process_file(filepath):
    filename = os.path.basename(filepath)
    extract_type = detect_type_from_filename(filename)
    
    if not extract_type:
        print(f"Skipping (no type match): {filename}")
        return
    
    print(f"Processing {filename} as type '{extract_type}'")
    
    try:
        df = pd.read_csv(filepath, low_memory=False)
        columns_to_keep = base_columns + extract_columns[extract_type]
        filtered_df = df[[col for col in columns_to_keep if col in df.columns]]
        
        output_path = os.path.join(os.path.dirname(filepath), f"filtered_{filename}")
        filtered_df.to_csv(output_path, index=False)
        print(f"Saved: {output_path}")
    except Exception as e:
        print(f"Failed to process {filename}: {e}")

def batch_process_folder(folder_path, region_keyword='global'):
    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.csv') and region_keyword in filename.lower():
            process_file(os.path.join(folder_path, filename))


run on folder of interest

In [25]:
input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs"

# Example usage:
if __name__ == "__main__":
    input_folder = input_folder  # Replace with your folder path
    batch_process_folder(input_folder, region_keyword="global")


Processing filtered_fixed_specific_global_railways_0_to_4176093.csv as type 'railways'
Saved: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs\filtered_filtered_fixed_specific_global_railways_0_to_4176093.csv
Processing fixed_specific_global_railways_0_to_4176093.csv as type 'railways'
Saved: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs\filtered_fixed_specific_global_railways_0_to_4176093.csv
Processing global_boat_access_0_to_724525.csv as type 'boat_access'
Failed to process global_boat_access_0_to_724525.csv: Error tokenizing data. C error: Expected 42 fields in line 11184, saw 47

Processing global_ferry_routes_0_to_31395.csv as type 'ferry_routes'
Failed to process global_ferry_routes_0_to_31395.csv: Error tokenizing data. C error: Expected 39 f

check geometry column in csv is valid

In [ ]:
def validate_and_fix_geometry(filepath):
    """
    Performs thorough validation and fixes WKT geometry strings in a CSV file for GEE upload.
    """
    print(f"Validating geometry in {os.path.basename(filepath)}...")
    
    try:
        # First check if file exists and is readable
        if not os.path.exists(filepath):
            print(f"Error: File does not exist: {filepath}")
            return filepath
            
        # Read file header to detect structure issues
        with open(filepath, 'r') as f:
            # Check first few lines to analyze structure
            header = f.readline().strip()
            sample_lines = [f.readline().strip() for _ in range(min(5, sum(1 for _ in f)))]
        
        # Check if header contains quoted fields
        has_quoted_header = '"' in header
        header_fields = header.replace('"', '').split(',')
        header_size = len(header_fields)
        
        print(f"Header has {header_size} columns")
        
        # Check if 'geometry' column exists
        if 'geometry' not in header_fields:
            print("Error: No 'geometry' column found in CSV")
            return filepath
            
        # Find geometry column index
        geom_index = header_fields.index('geometry')
        print(f"Geometry column found at index {geom_index}")
        
        # Analyze sample lines to detect issues
        line_field_counts = [len(line.split(',')) for line in sample_lines]
        if any(count != header_size for count in line_field_counts):
            print(f"Warning: Inconsistent field counts detected. Header: {header_size}, Lines: {line_field_counts}")
            print("This usually indicates unquoted geometry strings with commas")
        
        # Try reading with pandas with different options
        print("Attempting to read CSV with pandas...")
        try:
            # First try standard reading
            df = pd.read_csv(filepath)
            print("Successfully read CSV with standard options")
        except Exception as e1:
            print(f"Standard reading failed: {e1}")
            try:
                # Try with explicit quote character and escape
                df = pd.read_csv(filepath, quotechar='"', escapechar='\\', low_memory=False)
                print("Successfully read CSV with explicit quoting options")
            except Exception as e2:
                print(f"Reading with quoting options failed: {e2}")
                try:
                    # Last resort: read with Python's csv module
                    import csv
                    rows = []
                    with open(filepath, 'r') as f:
                        reader = csv.reader(f, quotechar='"', escapechar='\\')
                        headers = next(reader)
                        for row in reader:
                            # Ensure row matches header length
                            if len(row) >= len(headers):
                                rows.append(row[:len(headers)])
                            else:
                                # Pad row if it's too short
                                rows.append(row + [''] * (len(headers) - len(row)))
                    
                    df = pd.DataFrame(rows, columns=headers)
                    print("Successfully read CSV with csv module workaround")
                except Exception as e3:
                    print(f"All reading methods failed: {e3}")
                    return filepath
        
        # Check geometry column content
        total_rows = len(df)
        print(f"CSV has {total_rows} rows")
        
        # Define WKT validation patterns
        valid_wkt_patterns = [
            'POINT', 'LINESTRING', 'POLYGON', 'MULTIPOINT', 
            'MULTILINESTRING', 'MULTIPOLYGON', 'GEOMETRYCOLLECTION'
        ]
        
        # Function to check WKT validity
        def check_wkt_validity(geom_str):
            if not isinstance(geom_str, str):
                return False, "Not a string"
            
            # Basic pattern check
            if not any(geom_str.startswith(pattern) for pattern in valid_wkt_patterns):
                return False, "Not a valid WKT pattern"
            
            # Check for proper parentheses
            if geom_str.count('(') != geom_str.count(')'):
                return False, "Unbalanced parentheses"
                
            # For LineStrings, check for minimum points
            if geom_str.startswith('LINESTRING'):
                coords = geom_str.replace('LINESTRING', '').strip()[1:-1].split(',')
                if len(coords) < 2:
                    return False, "LineString has less than 2 points"
                    
            # For Polygons, check if first and last coordinates match
            if geom_str.startswith('POLYGON'):
                try:
                    # Extract first polygon ring
                    ring = geom_str.replace('POLYGON', '').strip()[2:-2].split(',')
                    if ring[0].strip() != ring[-1].strip():
                        return False, "Polygon ring not closed"
                except:
                    return False, "Invalid polygon format"
                    
            return True, "Valid"
        
        # Analyze geometry column
        geom_validity = []
        geom_errors = []
        
        for i, geom in enumerate(df['geometry']):
            is_valid, reason = check_wkt_validity(geom)
            geom_validity.append(is_valid)
            if not is_valid:
                geom_errors.append((i, reason, str(geom)[:50] + ('...' if len(str(geom)) > 50 else '')))
                if len(geom_errors) < 10:  # Limit error reporting
                    print(f"Row {i}: Invalid geometry - {reason}: {str(geom)[:50]}...")
        
        valid_count = sum(geom_validity)
        invalid_count = len(df) - valid_count
        
        print(f"Found {valid_count} valid and {invalid_count} invalid geometries")
        
        if invalid_count > 0:
            # Show error distribution
            error_types = {}
            for _, reason, _ in geom_errors:
                error_types[reason] = error_types.get(reason, 0) + 1
                
            print("\nError types:")
            for reason, count in error_types.items():
                print(f"  {reason}: {count} occurrences")
            
            # Fix file by ensuring quotes around geometry strings and fixing issues
            output_path = os.path.join(os.path.dirname(filepath), f"fixed_{os.path.basename(filepath)}")
            
            print(f"Writing fixed file to: {output_path}")
            
            # Write corrected CSV with geometry properly quoted and fixed
            with open(output_path, 'w', newline='') as outfile:
                # Write header
                outfile.write(','.join([f'"{col}"' for col in df.columns]) + '\n')
                
                fixed_count = 0
                # Process each row
                for _, row in df.iterrows():
                    values = []
                    for col, val in row.items():
                        # Special handling for geometry column
                        if col == 'geometry' and isinstance(val, str):
                            # Fix common geometry issues
                            fixed_val = val
                            
                            # Fix unbalanced parentheses
                            if fixed_val.count('(') > fixed_val.count(')'):
                                fixed_val = fixed_val + ')' * (fixed_val.count('(') - fixed_val.count(')'))
                                fixed_count += 1
                            
                            # Ensure LINESTRING and other patterns are properly formatted
                            for pattern in valid_wkt_patterns:
                                if pattern in fixed_val and not fixed_val.endswith(')'):
                                    fixed_val = fixed_val + ')'
                                    fixed_count += 1
                            
                            # Always quote geometry
                            values.append(f'"{fixed_val}"')
                        else:
                            # Quote strings containing commas
                            if isinstance(val, str) and (',' in val or '"' in val):
                                # Escape existing quotes and quote the whole thing
                                escaped_val = val.replace('"', '""')
                                values.append(f'"{escaped_val}"')
                            elif isinstance(val, str):
                                values.append(f'"{val}"')
                            else:
                                values.append(str(val) if pd.notna(val) else '')
                    
                    outfile.write(','.join(values) + '\n')
            
            print(f"Fixed {fixed_count} geometry issues")
            print(f"Fixed file saved to: {output_path}")
            return output_path
        else:
            print("All geometries appear valid.")
            return filepath
            
    except Exception as e:
        print(f"Error validating geometry: {e}")
        import traceback
        traceback.print_exc()
        return filepath

In [6]:
import os 
import pandas as pd
file_name = "global_railways_0_to_4176093.csv"
input_folder = r"C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs"
file_path = os.path.join(input_folder, file_name)
validate_and_fix_geometry(file_path)  # Call the validation function on the input folder

Validating geometry in global_railways_0_to_4176093.csv...
Header has 10 columns
Geometry column found at index 2
This usually indicates unquoted geometry strings with commas
Attempting to read CSV with pandas...


C:\Users\Arnell\AppData\Local\Temp\ipykernel_17928\3371152973.py:45: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


Successfully read CSV with standard options
CSV has 4176092 rows
Found 4176092 valid and 0 invalid geometries
All geometries appear valid.


'C:\\Users\\Arnell\\OneDrive - Food and Agriculture Organization\\project_work\\p0002_primary_forest_support\\work_in_progress\\osm_extracts\\osm_regional_250521_combined_csvs\\global_railways_0_to_4176093.csv'

In [7]:
def fix_coordinate_bleed(filepath):
    """
    Fixes CSV files where LINESTRING coordinates have bled into multiple columns.
    """
    print(f"Fixing coordinate bleed in {os.path.basename(filepath)}")
    
    # First read the raw file to analyze the structure
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    header = lines[0].strip().split(',')
    header_size = len(header)
    print(f"Header has {header_size} columns")
    
    # Check if 'geometry' column exists
    header_fields = [h.strip('"') for h in header]
    if 'geometry' not in header_fields:
        print("Error: No 'geometry' column found in CSV")
        return filepath
    
    # Find geometry column index
    geom_index = header_fields.index('geometry')
    print(f"Geometry column found at index {geom_index}")
    
    # Create output file path
    output_path = os.path.join(os.path.dirname(filepath), f"fixed_{os.path.basename(filepath)}")
    
    # Import csv module for more precise control
    import csv
    
    # Read the problematic file with csv module
    fixed_rows = []
    problem_count = 0
    
    with open(filepath, 'r', newline='') as infile:
        reader = csv.reader(infile)
        headers = next(reader)
        fixed_rows.append(headers)
        
        # Process each row
        for row_num, row in enumerate(reader, 1):
            if len(row) <= len(headers):
                # Row is fine, keep as is
                fixed_rows.append(row)
                continue
                
            # Row has too many fields - likely coordinate bleed
            problem_count += 1
            
            # Check if this looks like a line with coordinate bleed
            has_linestring = False
            for i, val in enumerate(row):
                if 'LINESTRING' in val:
                    has_linestring = True
                    linestring_start_idx = i
                    break
            
            if has_linestring:
                # Find where the LINESTRING should end (look for closing parenthesis)
                linestring_end_idx = None
                for i in range(linestring_start_idx, len(row)):
                    if ')' in row[i]:
                        linestring_end_idx = i
                        break
                
                if linestring_end_idx is not None:
                    # Reconstruct proper LINESTRING
                    linestring_parts = row[linestring_start_idx:linestring_end_idx+1]
                    full_linestring = ' '.join(linestring_parts).replace('" "', ', ')
                    
                    # Create fixed row
                    fixed_row = row[:linestring_start_idx]
                    fixed_row.append(full_linestring)
                    fixed_row.extend(row[linestring_end_idx+1:])
                    
                    # Ensure row has correct length
                    if len(fixed_row) < len(headers):
                        fixed_row.extend([''] * (len(headers) - len(fixed_row)))
                    elif len(fixed_row) > len(headers):
                        fixed_row = fixed_row[:len(headers)]
                    
                    fixed_rows.append(fixed_row)
                else:
                    # Can't find end of LINESTRING, keep original
                    fixed_rows.append(row[:len(headers)])
            else:
                # Just truncate to header length
                fixed_rows.append(row[:len(headers)])
    
    # Write fixed CSV
    print(f"Writing fixed file to: {output_path}")
    with open(output_path, 'w', newline='') as outfile:
        writer = csv.writer(outfile, quoting=csv.QUOTE_NONNUMERIC)
        writer.writerows(fixed_rows)
    
    print(f"Fixed {problem_count} rows with coordinate bleed")
    return output_path

In [8]:
# Fix a single file
fixed_file = fix_coordinate_bleed(file_path)

# Then validate it to ensure the fix worked
validated_file = validate_and_fix_geometry(fixed_file)

Fixing coordinate bleed in global_railways_0_to_4176093.csv
Header has 10 columns
Geometry column found at index 2
Writing fixed file to: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs\fixed_global_railways_0_to_4176093.csv
Fixed 0 rows with coordinate bleed
Validating geometry in fixed_global_railways_0_to_4176093.csv...
Header has 10 columns
Geometry column found at index 2
This usually indicates unquoted geometry strings with commas
Attempting to read CSV with pandas...


C:\Users\Arnell\AppData\Local\Temp\ipykernel_17928\3371152973.py:45: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


Successfully read CSV with standard options
CSV has 4176092 rows
Found 4176092 valid and 0 invalid geometries
All geometries appear valid.


In [12]:
pd.read_csv(file_path).head(20)  # Display the first few rows of the validated file

C:\Users\Arnell\AppData\Local\Temp\ipykernel_17928\4083411769.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(file_path).head(20)  # Display the first few rows of the validated file


,id,name,geometry,railway,tunnel,bridge,electrified,usage,service,layer
0,4064202,Apple Express (Avontuur Branch Line),"LINESTRING (25.4614787 -33.9603741, 25.4634427...",disused,NaN,NaN,no,NaN,NaN,NaN
1,4247103,NaN,"LINESTRING (18.4928136 -33.9326726, 18.4929916...",disused,NaN,NaN,no,NaN,spur,NaN
2,4251316,NaN,"LINESTRING (28.1946623 -25.7641181, 28.1950524...",rail,NaN,yes,contact_line,main,NaN,1.0
3,4251408,NaN,"LINESTRING (28.1684777 -25.7298785, 28.16846 -...",rail,NaN,NaN,contact_line,main,NaN,1.0
4,4251729,NaN,"LINESTRING (28.1998707 -25.7609415, 28.2002596...",rail,NaN,NaN,contact_line,main,NaN,NaN
5,4251762,NaN,"LINESTRING (28.218609 -25.7601207, 28.2189471 ...",rail,NaN,NaN,contact_line,main,NaN,NaN
6,4258125,NaN,"LINESTRING (28.2769996 -25.7164032, 28.2771463...",rail,NaN,NaN,no,NaN,spur,NaN
7,4258578,NaN,"LINESTRING (28.3077447 -25.7193714, 28.3080655...",rail,NaN,NaN,contact_line,main,NaN,NaN
8,4258695,NaN,"LINESTRING (28.3128881 -25.7258144, 28.3132916...",rail,NaN,NaN,no,NaN,spur,NaN
9,4258777,NaN,"LINESTRING (28.2937019 -25.7275168, 28.2943587...",rail,NaN,NaN,contact_line,main,NaN,NaN


In [14]:
def find_value_in_csv(filepath, search_value="Sterzing"):
    """
    Searches for a value in a CSV file and returns matching rows
    """
    print(f"Searching for '{search_value}' in {os.path.basename(filepath)}")
    
    # Try to read with pandas first
    try:
        df = pd.read_csv(filepath, low_memory=False)
        
        # Search in all string columns
        matches = pd.DataFrame()
        for col in df.columns:
            if df[col].dtype == 'object':  # Only search string columns
                col_matches = df[df[col].astype(str).str.contains(search_value, na=False)]
                if not col_matches.empty:
                    print(f"Found {len(col_matches)} matches in column '{col}'")
                    matches = pd.concat([matches, col_matches])
        
        if matches.empty:
            print(f"No matches found for '{search_value}'")
        else:
            print(f"Found {len(matches)} total matching rows")
            return matches
            
    except Exception as e:
        print(f"Error reading CSV with pandas: {e}")
        print("Trying with CSV module instead...")
        
        # Fall back to CSV module for problematic files
        import csv
        
        matches = []
        with open(filepath, 'r', newline='') as f:
            reader = csv.reader(f)
            headers = next(reader)
            
            for i, row in enumerate(reader, 1):
                for value in row:
                    if search_value in str(value):
                        print(f"Match in row {i}: {row[:3]}...")
                        matches.append((i, row))
                        break
        
        print(f"Found {len(matches)} matching rows")
        if matches:
            return matches



In [18]:
# Search in your file
matches = find_value_in_csv(fixed_file,search_value="11.4408977 46.8952226")
if isinstance(matches, pd.DataFrame) and not matches.empty:
    display(matches.head())

Searching for '11.4408977 46.8952226' in fixed_global_railways_0_to_4176093.csv
Found 1 matches in column 'geometry'
Found 1 total matching rows


,id,name,geometry,railway,tunnel,bridge,electrified,usage,service,layer
2275491,138615247,Vipiteno - Val di Vizze \nSterzing - Pfitsch,"LINESTRING (11.4408977 46.8952226, 11.4408773 ...",platform,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
def aggressive_fix_csv(filepath):
    """
    Applies a more aggressive fix to CSV file with LINESTRING coordinate issues.
    - Forces all geometry strings to be properly quoted
    - Ensures geometry strings with "LINESTRING" are properly formatted
    - Handles both coordinate bleed and other potential issues
    """
    print(f"Aggressively fixing CSV: {os.path.basename(filepath)}")
    
    # Import necessary modules
    import csv
    from io import StringIO
    
    output_path = os.path.join(os.path.dirname(filepath), f"fixed_aggressive_{os.path.basename(filepath)}")
    
    # Read the entire file first to analyze structure
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()
        
    # Check for common issues in content
    if 'LINESTRING' in content and '", "' in content:
        print("Detected quoted coordinates - this may indicate coordinate bleed")
    
    # Use csv module for precise control
    rows = []
    with open(filepath, 'r', newline='', encoding='utf-8', errors='replace') as infile:
        reader = csv.reader(infile)
        headers = next(reader)
        rows.append(headers)
        
        # Find geometry column index
        geom_idx = None
        for i, h in enumerate(headers):
            if h.strip('"').lower() == 'geometry':
                geom_idx = i
                break
                
        if geom_idx is None:
            print("Error: No geometry column found")
            return filepath
            
        print(f"Geometry column found at index {geom_idx}")
        problem_count = 0
        
        # Process each row
        for row_num, row in enumerate(reader, 1):
            # Handle rows with too many fields
            if len(row) > len(headers):
                problem_count += 1
                
                # Look for LINESTRING fragments
                linestring_idx = None
                for i, val in enumerate(row):
                    if 'LINESTRING' in val:
                        linestring_idx = i
                        break
                
                if linestring_idx is not None:
                    # Find all parts of the linestring and join them
                    linestring_parts = []
                    for i in range(linestring_idx, min(len(row), 100)):  # Limit search to 100 parts
                        linestring_parts.append(row[i])
                        if ')' in row[i]:
                            break
                            
                    # Reconstruct the geometry
                    geometry = ' '.join(linestring_parts).replace('" "', ', ')
                    
                    # Create a new row with fixed geometry
                    fixed_row = row[:linestring_idx]
                    fixed_row.append(geometry)
                    if linestring_idx + len(linestring_parts) < len(row):
                        fixed_row.extend(row[linestring_idx + len(linestring_parts):])
                        
                    # Ensure correct length
                    if len(fixed_row) > len(headers):
                        fixed_row = fixed_row[:len(headers)]
                    elif len(fixed_row) < len(headers):
                        fixed_row.extend([''] * (len(headers) - len(fixed_row)))
                        
                    rows.append(fixed_row)
                else:
                    # No LINESTRING found, just truncate
                    rows.append(row[:len(headers)])
            else:
                # Check if any field has unquoted LINESTRING
                fixed_row = list(row)  # Create a copy we can modify
                
                if geom_idx < len(row) and 'LINESTRING' in str(row[geom_idx]):
                    geom = row[geom_idx]
                    
                    # Check and fix common issues
                    if geom.count('(') != geom.count(')'):
                        fixed_row[geom_idx] = geom + ')' * (geom.count('(') - geom.count(')'))
                        problem_count += 1
                
                rows.append(fixed_row)
    
    # Write fixed CSV with all strings properly quoted
    with open(output_path, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile, quoting=csv.QUOTE_NONNUMERIC)
        writer.writerows(rows)
    
    print(f"Fixed {problem_count} problem rows")
    print(f"Saved to: {output_path}")
    
    return output_path

# Apply more aggressive fix
aggressive_fixed_file = aggressive_fix_csv(file_path)

# Validate it
problematic_rows_aggressive = find_problematic_rows(aggressive_fixed_file)

# Try to read with pandas to verify
try:
    df = pd.read_csv(aggressive_fixed_file)
    print(f"Successfully read aggressive fix with pandas. Found {len(df)} rows.")
except Exception as e:
    print(f"Error reading aggressive fix: {e}")

Aggressively fixing CSV: global_railways_0_to_4176093.csv
Geometry column found at index 2
Fixed 0 problem rows
Saved to: C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs\fixed_aggressive_global_railways_0_to_4176093.csv
Finding rows with column count mismatch in fixed_aggressive_global_railways_0_to_4176093.csv...
Header has 10 columns

Found 0 problematic rows out of 4176092 total rows

Column count distribution for problematic rows:


C:\Users\Arnell\AppData\Local\Temp\ipykernel_17928\1064488886.py:117: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(aggressive_fixed_file)


Successfully read aggressive fix with pandas. Found 4176092 rows.


In [22]:
def search_and_fix_specific_row(filepath, search_value="Sterzing - Pfitsch"):
    """
    Finds and fixes a specific problematic row
    """
    print(f"Searching for '{search_value}' in {filepath}")
    
    import csv
    
    output_path = os.path.join(os.path.dirname(filepath), f"fixed_specific_{os.path.basename(filepath)}")
    fixed = False
    
    # Read the file
    rows = []
    with open(filepath, 'r', newline='') as f:
        reader = csv.reader(f)
        headers = next(reader)
        rows.append(headers)
        
        for i, row in enumerate(reader, 1):
            if any(search_value in str(cell) for cell in row):
                print(f"Found match in row {i}")
                
                # Find LINESTRING fragment
                linestring_idx = None
                for j, val in enumerate(row):
                    if 'LINESTRING' in val:
                        linestring_idx = j
                        break
                
                if linestring_idx is not None:
                    print("Original row structure:")
                    print(f"  Row length: {len(row)}")
                    print(f"  LINESTRING at index {linestring_idx}: {row[linestring_idx]}")
                    
                    # Manually reconstruct this specific problematic geometry
                    geometry = "LINESTRING (11.4408977 46.8952226, 11.4408773 46.8952078, 11.4406561 46.89535, 11.4403906 46.8955204, 11.4403149 46.8955691, 11.4400695 46.8957268, 11.4398382 46.8958754, 11.4398128 46.895857, 11.439617 46.8959828, 11.439585 46.8960034, 11.4396308 46.8960366, 11.4408977 46.8952226)"
                    
                    # Create fixed row
                    fixed_row = row[:linestring_idx]
                    fixed_row.append(geometry)
                    
                    # Ensure correct length
                    while len(fixed_row) < len(headers):
                        fixed_row.append('')
                    
                    rows.append(fixed_row)
                    fixed = True
                    print("Manually fixed this specific row")
                else:
                    rows.append(row)
            else:
                rows.append(row)
    
    if fixed:
        # Write fixed CSV
        with open(output_path, 'w', newline='') as outfile:
            writer = csv.writer(outfile, quoting=csv.QUOTE_NONNUMERIC)
            writer.writerows(rows)
        print(f"Saved specific fix to: {output_path}")
        return output_path
    else:
        print("No matches found to fix")
        return filepath

# Try fixing specifically the problematic row
specific_fixed_file = search_and_fix_specific_row(file_path)


Searching for 'Sterzing - Pfitsch' in C:\Users\Arnell\OneDrive - Food and Agriculture Organization\project_work\p0002_primary_forest_support\work_in_progress\osm_extracts\osm_regional_250521_combined_csvs\global_railways_0_to_4176093.csv
Found match in row 2275492
Original row structure:
  Row length: 10
  LINESTRING at index 2: LINESTRING (11.4408977 46.8952226, 11.4408773 46.8952078, 11.4406561 46.89535, 11.4403906 46.8955204, 11.4403149 46.8955691, 11.4400695 46.8957268, 11.4398382 46.8958754, 11.4398128 46.895857, 11.439617 46.8959828, 11.439585 46.8960034, 11.4396308 46.8960366, 11.4408977 46.8952226)
Manually fixed this specific row
Found match in row 2275496
Original row structure:
  Row length: 10
  LINESTRING at index 2: LINESTRING (11.4384934 46.8969019, 11.4391173 46.8965308, 11.4417705 46.8948207, 11.4425789 46.8942653, 11.4425515 46.8942461, 11.4384677 46.8968839, 11.4384934 46.8969019)
Manually fixed this specific row
Saved specific fix to: C:\Users\Arnell\OneDrive - Food